Import Library

In [45]:
from pathlib import Path
import torch
import pandas as pd
import sys
import os

# go up two levels: Jupyter_notebook → scripts → project_root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)


# reuse functions from your script
# from Archeived.main_predict import get_model, get_dataset, run_pred
from mst.inference.predictor import load_model, get_dataset_class, predict_batch


import warnings
warnings.simplefilter("ignore", UserWarning)

Set Device and Path

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

run_dir = Path("/home/jovyan/work/MST/runs")
run_folder = Path("ODELIA/DinoV2ClassifierSlice_Final")

path_run = run_dir / run_folder

Load Model

In [47]:
model = load_model("DinoV2ClassifierSlice", path_run, device)
model.to(device)
model.eval()

Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main


DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

Load Test set

In [48]:
ds_test = get_dataset_class(name="ODELIA")(split="test")

In [49]:
print(ds_test.__dict__.keys())

dict_keys(['path_root', 'path_root_data', 'split', 'sequence', 'transform', 'df', 'item_pointers'])


In [50]:
target_uid = "02F4A1FB_left"

idx = ds_test.item_pointers.index(target_uid)
sample = ds_test[idx]

In [51]:
batch = {}
for k, v in sample.items():
    if torch.is_tensor(v):
        batch[k] = v.unsqueeze(0).to(device)
    else:
        batch[k] = v


In [52]:
batch

{'uid': '02F4A1FB_left',
 'source': tensor([[[[[-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            ...,
            [-0.5868, -0.6288, -0.6288,  ...,  1.1775,  0.9464,  0.9464],
            [-0.5868, -0.6078, -0.6498,  ...,  0.9254,  0.8624,  0.9674],
            [-0.6078, -0.6078, -0.6288,  ...,  0.8414,  0.8414,  0.9884]],
 
           [[-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            [-0.6288, -0.6288, -0.6288,  ..., -0.6288, -0.6288, -0.6288],
            ...,
            [-0.6078, -0.5448, -0.4818,  ...,  1.1565,  0.9044,  0.9044],
            [-0.5868, -0.5028, -0.4818,  ...,  0.9884,  0.7574,  0.7364],
            [-0.5028, -0.4397, -0.4818,  ...,  1.0514,  0.6314,  0.5264]],
 
           [[-0.6288, -0.6288, -0.628

In [73]:
pred = predict_batch(model, batch, device=device).cpu()
#probs = torch.softmax(pred, dim=-1).cpu()
pred_class = torch.argmax(pred, dim=1)

print(f"Pred is {pred}")
print(f"Predicted class is {pred_class}")

print("GT:", batch["target"].item())
print("Predicted class:", pred_class.item())
print("Probabilities:", pred.squeeze().numpy())


Pred is tensor([[4.0925e-01, 5.5058e-04, 5.9020e-01]])
Predicted class is tensor([2])
GT: 0
Predicted class: 2
Probabilities: [4.0924796e-01 5.5057689e-04 5.9020138e-01]


In [72]:
logits = model(batch["source"].to(device))
prob = torch.softmax(logits, dim=-1)#[0]
predicted_class = logits.argmax(dim=-1).item()

print(f"Logits: {logits}")
print(f"Probabilities: {prob}")
print(f"Predicted class from logits: {predicted_class}")

Logits: tensor([[ 2.1233, -4.4878,  2.4894]], grad_fn=<AddmmBackward0>)
Probabilities: tensor([[4.0925e-01, 5.5058e-04, 5.9020e-01]], grad_fn=<SoftmaxBackward0>)
Predicted class from logits: 2


In [64]:
prob.detach().cpu().tolist()

[0.4092479646205902, 0.0005505768931470811, 0.5902013778686523]